# Module 1.3 -- Ray: Heterogeneous AI Compute at Scale

**Track:** Big Data Engineering for AI Systems  
**Toolchain:** Ray Core - Ray Data - Ray Train - vLLM  
**Objective:** Scale Python AI workloads from laptop to 1000-GPU clusters using Ray's task and actor model.

---

## When Do You Need Ray? (And When Is Spark Enough?)

| Workload | Best Tool | Why |
|----------|----------|-----|
| SQL analytics on terabytes | **Spark** | Optimized for structured data |
| ETL pipelines (Bronze/Silver/Gold) | **Spark** | Declarative pipelines, Delta Lake |
| Training a 70B LLM on 128 GPUs | **Ray** | Native GPU scheduling, NCCL |
| Batch inference on 1M images | **Ray** | Pipelined CPU+GPU execution |
| Reinforcement learning | **Ray** | Stateful actors, RLlib |
| Feature engineering (tabular) | **Spark** | SQL + window functions |

### The Core Difference

**Spark** = Data parallelism (same operation on every partition)  
**Ray** = Task parallelism (different operations on different resources)  

**Analogy:** Spark is a factory assembly line (every station does the same thing). Ray is a hospital (doctors, nurses, lab techs do different things simultaneously).


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Spark is great for data, but terrible for deeply nested Python objects and GPU tasks. Seniors use Ray as a universal distributed compute engine for ML training, hyperparameter tuning, and batch inference.

**Mental Model:** Ray takes standard Python functions and translates them invisibly into tasks executed across thousands of CPUs and GPUs, handling all the network communication (RPCs) in the background.

**Common Junior Pitfall:** Passing giant objects (like a 2GB dataset) as an argument to a Ray task 100 times, causing massive network saturation. Seniors pass object references (`ray.put()`) instead.

---


In [ ]:
# Cell 1 -- Install
!pip install -q ray[data] numpy pandas


---
## 1 - Ray Architecture: Tasks and Actors

### Tasks vs Actors

| | Task | Actor |
|---|---|---|
| State | Stateless (fresh each call) | Stateful (remembers between calls) |
| Use case | Pure functions, embarrassingly parallel | Models, database connections |
| Syntax | `@ray.remote` on function | `@ray.remote` on class |

### The Ray Execution Model

```
Your Python code
       |
  Ray Driver (your laptop)
       |
  Ray Head Node (scheduler)
      / | \
 Worker Worker Worker  (CPU/GPU nodes)
```


In [ ]:
# Cell 2 -- Ray tasks and actors demo
import ray
import numpy as np
import time

# 🎓 SENIOR TIP: In production, never use num_cpus=4. Connect to `ray://head-node-ip:10001` where\n# 100s of GPU workers wait.\n# 🎓 SENIOR TIP: In production, never use num_cpus=4. Connect to `ray://head-node-ip:10001` where\n# 100s of GPU workers wait.\nray.init(ignore_reinit_error=True, num_cpus=4)

# TASK: stateless parallel function
@ray.remote
def process_batch(batch_id, data):
    """Simulate processing a data batch (e.g., feature extraction)."""
    time.sleep(0.1)  # Simulate work
    return {'batch_id': batch_id, 'mean': float(np.mean(data)), 'std': float(np.std(data))}

# Launch 10 tasks in PARALLEL (Ray distributes across workers)
# 🎓 SENIOR TIP: .remote() returns a 'Future' immediately. The data generation happens asynchronously.\n# 🎓 SENIOR TIP: .remote() returns a 'Future' immediately. The data generation happens asynchronously.\nfutures = [process_batch.remote(i, np.random.randn(1000)) for i in range(10)]
# 🎓 SENIOR PITFALL: Calling ray.get() blocks execution. Do not call this inside a loop! Gather all futures first.\n# 🎓 SENIOR PITFALL: Calling ray.get() blocks execution. Do not call this inside a loop! Gather all futures first.\nresults = ray.get(futures)

print('Ray Parallel Tasks:')
for r in results[:5]:
    print(f'  Batch {r["batch_id"]}: mean={r["mean"]:.3f}, std={r["std"]:.3f}')
print(f'  Total: {len(results)} batches processed in parallel')
\n

In [ ]:
# Cell 3 -- Ray Actor: stateful model server
@ray.remote
class ModelServer:
    """Stateful actor that holds a model in GPU memory across calls."""
    def __init__(self, model_name):
        self.model_name = model_name
        self.request_count = 0
        # In production: self.model = load_model(model_name)

    def predict(self, input_data):
        self.request_count += 1
        # Simulated inference
        return {'model': self.model_name, 'prediction': float(np.mean(input_data)), 'request_num': self.request_count}

    def get_stats(self):
        return {'model': self.model_name, 'total_requests': self.request_count}

# Create actor (lives in a worker, keeps state between calls)
# 🎓 SENIOR PITFALL: Creating an Actor ties up a whole cluster worker core exclusively. Use sparingly!\n# 🎓 SENIOR PITFALL: Creating an Actor ties up a whole cluster worker core exclusively. Use sparingly!\nserver = ModelServer.remote('my-llm-7b')

# Send multiple requests to the SAME actor (model stays loaded)
preds = ray.get([server.predict.remote(np.random.randn(100)) for _ in range(5)])
stats = ray.get(server.get_stats.remote())

print('Ray Actor (Stateful Model Server):')
for p in preds:
    print(f'  Request #{p["request_num"]}: prediction={p["prediction"]:.3f}')
print(f'  Total served: {stats["total_requests"]}')
\n

---
## 2 - Ray Data: Scalable Batch Inference Pipeline

### The LLM Batch Inference Problem

You have 1 million documents to process through an LLM. Sequential processing takes days. You need:
1. **CPU workers** to preprocess (tokenize, normalize)
2. **GPU workers** to run inference (the LLM)
3. **CPU workers** to postprocess (parse output, save)

Ray Data **pipelines** these stages so GPUs never idle waiting for CPU preprocessing.

```
[CPU: Preprocess] --> [GPU: LLM Inference] --> [CPU: Postprocess]
     batch 3              batch 2                 batch 1
```


In [ ]:
# Cell 4 -- Ray Data batch inference pipeline
import ray.data

# Create a dataset of documents
docs = [{'text': f'Document {i}: This is sample text for batch inference.', 'doc_id': i} for i in range(100)]
ds = ray.data.from_items(docs)

# Stage 1: Preprocess (runs on CPU)
def preprocess(batch):
    batch['token_count'] = [len(t.split()) for t in batch['text']]
    batch['text_clean'] = [t.lower().strip() for t in batch['text']]
    return batch

# Stage 2: Inference (would run on GPU in production)
def mock_inference(batch):
    batch['embedding'] = [np.random.randn(384).tolist() for _ in batch['text']]
    batch['sentiment'] = [np.random.choice(['positive','negative','neutral']) for _ in batch['text']]
    return batch

# Stage 3: Postprocess
def postprocess(batch):
    batch['summary'] = [f'Doc {did}: {tc} tokens, {s}' for did, tc, s in zip(batch['doc_id'], batch['token_count'], batch['sentiment'])]
    return batch

# Execute pipeline (Ray handles parallelism and pipelining)
result = ds.map_batches(preprocess).map_batches(mock_inference).map_batches(postprocess)
output = result.take(5)

print('Ray Data Batch Inference Pipeline:')
for row in output:
    print(f'  {row["summary"]}')
print(f'  Processed {ds.count()} documents through 3-stage pipeline')


---
## 3 - Ray Direct Transport (RDT): Zero-Copy GPU Communication

### The Historical Bottleneck

```
GPU 1 --> serialize --> CPU RAM --> Ray Object Store --> CPU RAM --> deserialize --> GPU 2
         (SLOW!)         (SLOW!)        (SLOW!)          (SLOW!)      (SLOW!)
```

### RDT: Bypass Everything

```
GPU 1 ═══════ RDMA/NCCL direct transfer ═══════> GPU 2
              (zero copy, zero serialization)
```

| | Traditional | RDT |
|---|---|---|
| Transfer path | GPU->CPU->Store->CPU->GPU | GPU->GPU direct |
| Serialization | Required | None |
| Speed | 1x baseline | Up to 1000x faster |
| Use case | General data | Tensor/model parallel training |


In [ ]:
# Cell 5 -- RDT configuration reference
rdt_config = '''
# Enable Ray Direct Transport in ray cluster config
# ray_cluster.yaml
ray:
  transport:
    type: "direct"  # Enable RDT
    backends:
      - nccl       # NVIDIA NCCL for GPU-GPU on same node
      - gloo       # Gloo for CPU or cross-node
      - rdma       # InfiniBand RDMA for cross-node GPU-GPU

# In Python code:
import ray
ray.init(
    runtime_env={"env_vars": {"RAY_EXPERIMENTAL_DIRECT_TRANSPORT": "1"}},
)

# Transfers between GPU actors now bypass CPU entirely!
'''
print('Ray Direct Transport Configuration:')
print(rdt_config)
print('Performance comparison:')
print('  Traditional: GPU->CPU->Store->CPU->GPU = ~2 GB/s')
print('  RDT (NCCL):  GPU->GPU direct           = ~300 GB/s (NVLink)')
print('  RDT (RDMA):  GPU->GPU cross-node       = ~100 GB/s (InfiniBand)')


---

## Production Reality Check

| In This Notebook | In Production |
|---|---|
| `ray.init(num_cpus=4)` local | Ray cluster with 100s of GPU nodes |
| `np.random.randn()` fake inference | Real vLLM/TensorRT-LLM engines |
| `time.sleep(0.1)` simulated work | Actual GPU compute (seconds per batch) |

### Real Ray Cluster Architecture

```
KubeRay Operator (manages lifecycle)
       |
  Ray Head Node (scheduler, GCS)
      / | \
 Worker Worker Worker  (each with 8x A100 GPUs)
```

### When Ray Gets Complicated

| Challenge | Description |
|-----------|-------------|
| GPU memory | Models don't fit on one GPU -> tensor parallelism |
| Fault tolerance | Worker dies mid-training -> checkpoint + restart |
| Autoscaling | Scale 0->100 GPUs in minutes -> KubeRay + cloud APIs |
| Cost | A100 GPU: $2-4/hr. 100 GPUs for 1 day = $4,800-9,600 |


---
## Summary

| Concept | What You Learned |
|---------|------------------|
| Spark vs Ray | Data parallelism vs task parallelism |
| Ray Tasks | Stateless parallel functions |
| Ray Actors | Stateful services (model servers) |
| Ray Data | Pipelined CPU+GPU batch inference |
| RDT | Zero-copy GPU-to-GPU communication |

**Next -->** `08_model_development.ipynb` -- Training Models with Tracked Experiments